# **Import Libraries**

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
import itertools

# File Path
model_performance_file = "/content/ADASYN_KFold_Formate_Result(INCART 2-lead Arrhythmia Database).xlsx"
data_reader = pd.ExcelFile(model_performance_file)
all_metrics = data_reader.sheet_names

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# **Topsis**

In [3]:
def compute_model_priority(mean_data):
    # Normalization
    val_array = mean_data.values
    normalization_factor = np.linalg.norm(val_array)
    normalized_vals = val_array / (normalization_factor + 1e-10)

    # Ideal Solutions
    target_best = 1.0
    target_worst = 0.0

    dist_to_best = np.abs(normalized_vals - target_best)
    dist_to_worst = np.abs(normalized_vals - target_worst)

    # Final TOPSIS Score
    priority_scores = dist_to_worst / (dist_to_best + dist_to_worst)
    return priority_scores

topsis_summary_list = []

for sheet in all_metrics:
    print(f"\n>>> Ranking Models for: {sheet}")
    raw_df = pd.read_excel(data_reader, sheet_name=sheet)

    # Model names
    target_models = [col for col in raw_df.columns if col.lower() != 'fold']


    avg_performance = raw_df.iloc[:10][target_models].mean()

    calculated_scores = compute_model_priority(avg_performance)

    sheet_result = pd.DataFrame({
        'Metric': sheet,
        'Model': target_models,
        'Performance_Score': calculated_scores,
        'Raw_Mean': avg_performance.values
    })

    # Rank
    sheet_result['Position'] = sheet_result['Performance_Score'].rank(ascending=False)
    sheet_result = sheet_result.sort_values('Position').reset_index(drop=True)

    print(sheet_result[['Metric', 'Model', 'Performance_Score', 'Position']])
    topsis_summary_list.append(sheet_result)

# Excel
pd.concat(topsis_summary_list).to_excel('topsis_Rank_INCART 2-lead Arrhythmia Database(12 models).xlsx', index=False)


>>> Ranking Models for: Accuracy
      Metric                Model  Performance_Score  Position
0   Accuracy                  KNN           0.311994       1.0
1   Accuracy                  MLP           0.311803       2.0
2   Accuracy             LightGBM           0.311341       3.0
3   Accuracy                  GBM           0.311341       4.0
4   Accuracy        Random Forest           0.311310       5.0
5   Accuracy        Decision Tree           0.310213       6.0
6   Accuracy  Stacking Classifier           0.309732       7.0
7   Accuracy              xgboost           0.309194       8.0
8   Accuracy             CatBoost           0.296840       9.0
9   Accuracy              Bagging           0.223722      10.0
10  Accuracy  Logistic Regression           0.211480      11.0
11  Accuracy                  SVM           0.209768      12.0

>>> Ranking Models for: Precision
       Metric                Model  Performance_Score  Position
0   Precision                  KNN           0.3

# **Wilcoxon**

In [4]:
wilcoxon_final_list = []

for sheet in all_metrics:
    print(f"\n" + "-"*60)
    print(f" STATISTICAL SIGNIFICANCE (WILCOXON): {sheet} ")
    print("-"*60)

    full_df = pd.read_excel(data_reader, sheet_name=sheet)

    fold_data = full_df.iloc[:10]
    available_models = [col for col in fold_data.columns if col.lower() != 'fold']

    pairwise_stats = []

    for m1, m2 in itertools.combinations(available_models, 2):
        group_a = fold_data[m1]
        group_b = fold_data[m2]

        try:
            stat, p_val = wilcoxon(group_a, group_b, nan_policy='omit')
            pairwise_stats.append({
                "Evaluation_Metric": sheet,
                "Refference Model": m1,
                "Compared Model": m2,
                "P_Value": round(p_val, 6),
                "Significant": p_val < 0.05
            })
        except:
            pass

    comparison_df = pd.DataFrame(pairwise_stats)
    print(comparison_df)
    wilcoxon_final_list.append(comparison_df)

pd.concat(wilcoxon_final_list).to_excel('Wilcoxon_INCART 2-lead Arrhythmia Database(12 models).xlsx', index=False)


------------------------------------------------------------
 STATISTICAL SIGNIFICANCE (WILCOXON): Accuracy 
------------------------------------------------------------
   Evaluation_Metric     Refference Model       Compared Model   P_Value  Significant
0           Accuracy  Logistic Regression        Decision Tree  0.001953         True
1           Accuracy  Logistic Regression        Random Forest  0.001953         True
2           Accuracy  Logistic Regression                  GBM  0.001953         True
3           Accuracy  Logistic Regression              xgboost  0.001953         True
4           Accuracy  Logistic Regression             LightGBM  0.001953         True
5           Accuracy  Logistic Regression             CatBoost  0.001953         True
6           Accuracy  Logistic Regression                  SVM  0.001953         True
7           Accuracy  Logistic Regression                  KNN  0.001953         True
8           Accuracy  Logistic Regression              